In [ ]:
import os, sys
cur_directory = os.getcwd()
project_folder = os.path.abspath(os.path.join(cur_directory, os.pardir, os.pardir))
save_directory = os.path.join(cur_directory, 'saved')
sys.path.append(os.path.join(project_folder, 'analysis-suite'))

In [ ]:
import numpy as np
import matplotlib as mpl
from matplotlib import pyplot as plt
import analyses as als
import datasets as ds
import models as mds
import tools
from random import sample
from tqdm import tqdm
from argparse import Namespace
from time import gmtime, strftime
import pickle

## Params

In [ ]:
V_width = 1
V_E_buffer_array = np.arange(30).astype(int)
E_width = 1

N_meta_neurons = 20 # empirically better performance than K=N when at high-iterations
N_training_repeats = 10
mov_ind_array = np.arange(5)  # bushes, water, leaves, fish, opticflow
critic_params = {
    "hidden_dim": 32,
    "embed_dim": 30,
    "layers": 2,
    "activation": "tanh",
}
opt_params = {"track_quantities": False, "quiet": True, "iterations": 16}
mi_measurement_params = {"quiet": True, "batch_size": 500, "N_batches": 500}

train_fraction = 0.5
samp_keep = 1

# shuffle time bins in a way that is the same across movie repeats
shuffle_time = False

# shuffle trial index for each timestep
shuffle_trial_per_time = False

# Put variable names here for parameters we want saved.
save_vars = {
    "mi_measurement_params",
    "N_training_repeats",
    "N_meta_neurons",
    "critic_params",
    "opt_params",
    "V_width",
    "V_E_buffer_array",
    "E_width",
    "train_fraction",
    "samp_keep",
    "shuffle_time"
}


In [ ]:
V_E_buffer_array

In [ ]:
1.4 * 1.2

## Run analysis and save

In [ ]:
for mov_ind in mov_ind_array:
    # Prepare data
    data_path = os.path.join(project_folder, 'data', 'nat_scenes_data')
    D = ds.DataNatScenes(data_path, mov_ind=mov_ind)
    mov_ind = D.mov_ind
    mov_name = D.movname
    save_vars.add('mov_ind')

    # data_path = os.path.join(project_folder, 'data', 'bar_stim_data')
    # D = ds.DataBar(data_path)
    # mov_name = 'bar'

    save_vars.add('mov_name')
    print(f'current stimulus is {mov_name}')

    # Run analysis
    experiment_dims = (len(V_E_buffer_array), N_training_repeats)
    mi_averages_test = np.zeros(experiment_dims)
    mi_averages_train = np.zeros(experiment_dims)
    mi_stdevs_train = np.zeros(experiment_dims)
    mi_stdevs_test = np.zeros(experiment_dims)

    train_points = np.zeros(experiment_dims, dtype=object)
    test_points = np.zeros(experiment_dims, dtype=object)
    save_vars.update(
        {
            "mi_averages_train",
            "mi_averages_test",
            "mi_stdevs_train",
            "mi_stdevs_test",
            "train_points",
            "test_points",
        }
    )

    for i, V_E_buffer in tqdm(enumerate(V_E_buffer_array)):
        for training_repeat in range(N_training_repeats):
        
            trial_inds = (i, training_repeat)

            # Domains, training and test subsets
            D.reset()
            D.trim(60)
            if shuffle_time:
                D.shuffle_time_and_trial()
        
            if shuffle_trial_per_time:
                D.shuffle_trial_per_time()


            V, E = D.create_V_and_E_datasets(V_width, V_E_buffer, E_width, samp_keep)
            (
                current_train_points,
                current_test_points,
            ) = tools.generate_train_test_indices(V.P, train_fraction)
            train_points[trial_inds] = current_train_points
            test_points[trial_inds] = current_test_points

            # solve IB for training set points
            coarse_grainer = mds.PolynomialLayer(output_dim=N_meta_neurons, order=1)
            critic_function = mds.SeparableCritic(**critic_params)

            # Train current
            V.subset_points(current_train_points)
            E.subset_points(current_train_points)
            als.IB(
                X=V.X,
                Y=E.X,
                coarse_grainer=coarse_grainer,
                critic_function=critic_function,
                **opt_params
            )
        
            # Measure mutual information
            # Measure InfoNCE for train
            T = coarse_grainer(V.X.T).numpy().T
            mi_train = als.MI_NCE(
                X=T, Y=E.X, critic_function=critic_function, **mi_measurement_params
            )
            mi_averages_train[trial_inds] = mi_train["mi"]
            mi_stdevs_train[trial_inds] = mi_train["mi_stdev"]

            # Measure InfoNCE for test
            V.reset()
            E.reset()
            V.subset_points(current_test_points)
            E.subset_points(current_test_points)
            T = T = coarse_grainer(V.X.T).numpy().T
            mi_test = als.MI_NCE(
                X=T, Y=E.X, critic_function=critic_function, **mi_measurement_params
            )
            mi_averages_test[trial_inds] = mi_test["mi"]
            mi_stdevs_test[trial_inds] = mi_test["mi_stdev"]

    # Select variables to save
    # select variables
    coarse_grainer_type = type(coarse_grainer)
    critic_type = type(critic_function)
    save_vars.update({"critic_type", "coarse_grainer_type"})
    local_dict = locals()
    output_data = {k: local_dict[k] for k in save_vars}

    # Save
    # save things
    timestamp = strftime("%Y.%m.%d %Hh %Mm %Ss.dat",gmtime())
    # full_file_path = os.path.join(save_directory, strftime("%Y.%m.%d %Hh %Mm %Ss",gmtime())+'.dat')
    if shuffle_time:
        prepend = "(shuffled) "
    elif shuffle_trial_per_time:
        prepend = "(shuffled trial per time)"
    else:
        prepend = ""

    stim = mov_name
    full_file_path = os.path.join(save_directory, stim, prepend + 'all neurons, high-iter.dat')
    # full_file_path = os.path.join(save_directory, stim, prepend + 'all neurons, trim 60.dat')
    print(f"data will be saved to:\n{full_file_path}")

    # 
    os.makedirs(os.path.dirname(full_file_path), exist_ok=True)
    with open(full_file_path, 'wb+') as f:
                pickle.dump(output_data, f)
    tools.record_parameters(full_file_path, output_data)


## Reload and plot

In [ ]:
# stim = 'bar'
# stim = 'opticflow'
stim = 'bushes'

In [ ]:
load_path = project_folder + f'/analyses/info decay timescale/saved/{stim}/all neurons, high-iter.dat'
with open(load_path, 'rb') as f:
    data_dict = pickle.load(f)
data = Namespace(**data_dict)


In [ ]:
data_dict.keys()

In [ ]:
plt.plot(data.mi_averages_train)